Comparing LLaMA model response quality between 4-bit quantized and full precision inference using Hugging Face Transformers on a T4 GPU to evaluate the trade-off between memory efficiency and output accuracy. This code to be ran on google colab T4 GPU machine


In [ ]:
#command used to install and update python packages (bitsandbytes for GPU memory usage , accellarate by hugging face for running models accros gpus,cpus etc and transformers also by hugging face library for NLP)
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
#importing modules required sdda
from google.colab import userdata
from huggingface_hub import login, InferenceClient
#Autotokenizer - automatically loads the correct tokenizer to convert text into model readable tokens and back
#AutomModelForCausalLM -loads a pretrained causal language model used for text generation likeGPT
#TextStreamer - streams generated text in real time as the model produces it(writing while thinking)
#BitsAndBytesConfig - configures model loading with memory-efficient quantization settings
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
import pandas as pd


In [ ]:
#retrieving my hugging face token , and authenticating using hugging face login function
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
#instantiating a llama hugging face model
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [ ]:
#defining a system prompt for the bot
system_prompt = """
You are a highly precise and structured assistant.

Follow these rules strictly:
1. Break down your answer into clear numbered steps.
2. Ensure all reasoning is logically sound and easy to follow.
3. Avoid vague statements — be specific and accurate.
4. Use concise language with no unnecessary words.
5. After the explanation, provide a one-sentence summary.
6. If solving a problem, show all calculations clearly.
7. Do not skip steps or make assumptions.

Your output must be deterministic, structured, and correct.

"""
user_prompt = """
A company has 3 departments: A, B, and C.

- Department A has twice as many employees as B.
- Department C has 10 fewer employees than A.
- The total number of employees is 130.

1. Determine how many employees are in each department.
2. Show all steps clearly.
3. Verify your final answer.
4. Summarize the result in one sentence

"""


In [ ]:
#creating list of dictionary messages that structure the converstion  by assigning roles and their corresponding propmpt content for a chat-based model
messages = [
    {"role": "user", "content":user_prompt},
    {"role": "system", "content": system_prompt}
  ]

In [ ]:
#creating a configuration that tells the model to load using 4-bit quantization wit optimized settings to reduce memory usage while maintainig performance
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="fp4"
    #llm_int8_skip_modules=["lm_head"]
)

In [ ]:
#here I am passing the messages into the chat model and getting response
client = InferenceClient(model = LLAMA,token = hf_token)
response = client.chat.completions.create(messages = messages)
#response.choices[0].message.content


In [ ]:

#loading the correct tokenizer for model, set its padding token to match the end of sequence 
# token(to avoid padding issue during generation), then format chat msgs into tensors and convert tensors into 
# pytorch format and move them to gpu for efficient processing


tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
# loading a pretrained causal language model with automatic device placement and quantization 
# settings, then calculating and printing how much memory the model occupies in megabytes
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

In [ ]:
#methods that will be used to compare the results of quantized and non quantized answer to a stress reasoning prompt
def run_nonQuant(model_name):

  #non qunatized
  output_full = response.choices[0].message.content


  return output_full

def run_Quant(model_name):

  #quantized
  #trim the output to only get the response from the model
  output_quant = model.generate(inputs, max_new_tokens=600)
  generated_tokens = output_quant[0][inputs.shape[-1]:]
  quant = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  

  return quant




In [ ]:
#decode the tokens to readable text
Quant = run_Quant(LLAMA)
Full = run_nonQuant(LLAMA)

In [ ]:
#creating table to display
df = pd.DataFrame({
    "Model" : ['Non Quantized Output', 'Quantized Output'],
    "Output" : [Full, Quant]
})
display(df)